# Data exploration
Quick look at the gait parquet files before building features.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
gaits = ['normal', 'antalgic', 'stiff_legged', 'lurch', 'steppage', 'trendelenburg']
df = pd.concat([pd.read_parquet(f'../data/parquet/{g}.parquet') for g in gaits], ignore_index=True)
df.shape

In [ ]:
df.iloc[:, :10].head()

First 8 columns are metadata + time, the rest are 25 joints x (x, y, z).

In [ ]:
df.dtypes.value_counts()

### Class balance

In [ ]:
df.groupby('gait_label')['source_file'].nunique()

In [ ]:
df.groupby('gait_label')['subject_id'].nunique()

Each walk has several sensor files, so one 'sample' later = one sensor file.

In [ ]:
df['sensor_id'].value_counts()

In [ ]:
df.groupby('source_file')['n_frames'].first().describe()

### Bad values
The build step warned about huge numbers and some NaNs, checking what is left.

In [ ]:
coords = df.filter(regex='^joint')
coords.isna().sum().sum()

In [ ]:
bad = coords.isna().any(axis=1) | (coords.abs() > 10).any(axis=1)
bad.sum()

In [ ]:
df[bad]['source_file'].nunique()

### A quick look at the signal

In [ ]:
walk = df[df['source_file'] == df['source_file'].iloc[0]]
plt.plot(walk['time'], walk['joint17_y'])
plt.show()

In [ ]:
df[~bad].groupby('gait_label')['joint14_y'].mean()

### Notes
- classes look roughly balanced
- a few files have broken or NaN coordinates, dropping those in the features step
- 6 sensors per walk, so the split has to be by subject
- different gaits already show different joint heights, so features should work